In [ ]:
#!/usr/bin/env python3
"""
Cross‑Asset Order‑Flow Imbalance (OFI) Feature Selection
=======================================================

Construct the **cross‑asset best‑level OFI matrix** (Eq. 8) and, over
non‑overlapping 30‑minute windows, apply **LASSO** to select the tickers
whose imbalances are most predictive of the *self* ticker’s one‑minute
return (Eq. 9) in *“Cross‑Impact of Order‑Flow Imbalance”*.

---------------------------------------------------------------------------
Directory Layout (expected)
---------------------------------------------------------------------------
ofi_data/
    AAPL_ofi.csv
    MSFT_ofi.csv
    TSLA_ofi.csv
    ...
    AAPL_price.csv      ← close prices for the self ticker
Each *_ofi.csv* must contain:
    ts_event,  best_level_ofi
Each *_price.csv* must contain:
    ts_event,  close

---------------------------------------------------------------------------
Output
---------------------------------------------------------------------------
selected_cross_ofi.csv
    window_start, window_end, self_ticker,
    selected_tickers (list[str]), coef_values (list[float])

---------------------------------------------------------------------------
Author  : ChatGPT “OFI Builder”
"""

from __future__ import annotations

from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV


# ── CONFIGURE PATHS & PARAMETERS ────────────────────────────────────────────
DATA_DIR      = Path("ofi_data")                 # folder with *_ofi.csv files
SELF_TICKER   = "AAPL"                           # ticker we predict
PRICE_CSV     = DATA_DIR / f"{SELF_TICKER}_price.csv"

WINDOW_MIN    = 30                               # rolling window length
LASSO_STATE   = 42                               # reproducibility
OUT_CSV       = Path("selected_cross_ofi.csv")   # final summary file

# Column conventions (keep consistent with earlier scripts)
COL_TIME      = "ts_event"
COL_OFI       = "best_level_ofi"                 # OFI value column in *_ofi.csv
COL_CLOSE     = "close"                          # close price for returns


# ── HELPER FUNCTIONS ────────────────────────────────────────────────────────
def load_cross_ofi(folder: Path) -> pd.DataFrame:
    """
    Load every *_ofi.csv file and align them into a single DataFrame.

    Returns
    -------
    pandas.DataFrame
        Index:   ts_event (DatetimeIndex, UTC)
        Columns: one per ticker (ticker symbol upper‑case).
        NaNs are filled with 0.0 (no imbalance recorded).
    """
    frames = []
    for fp in sorted(folder.glob("*_ofi.csv")):
        ticker = fp.stem.replace("_ofi", "")
        df_tmp = pd.read_csv(
            fp,
            usecols=[COL_TIME, COL_OFI],
            parse_dates=[COL_TIME],
            index_col=COL_TIME,
        ).rename(columns={COL_OFI: ticker})
        frames.append(df_tmp)

    if not frames:
        raise FileNotFoundError(f"No '*_ofi.csv' files found in {folder}")

    df_ofi = pd.concat(frames, axis=1).sort_index().fillna(0.0)
    return df_ofi


def load_self_returns(price_path: Path) -> pd.Series:
    """
    Compute one‑minute arithmetic returns for the SELF_TICKER.

    Missing minutes (if any) are forward‑filled then diff‑pct‑changed.
    """
    df_price = pd.read_csv(
        price_path,
        usecols=[COL_TIME, COL_CLOSE],
        parse_dates=[COL_TIME],
        index_col=COL_TIME,
    ).sort_index()

    # Ensure 1‑minute regularity by re‑indexing on full union
    full_idx = pd.date_range(df_price.index[0], df_price.index[-1], freq="1min", tz="UTC")
    df_price = df_price.reindex(full_idx).ffill()

    returns = df_price[COL_CLOSE].pct_change().fillna(0.0).rename("y")
    return returns


def rolling_windows(idx: pd.DatetimeIndex, minutes: int) -> List[Tuple[pd.Timestamp, pd.Timestamp]]:
    """Yield non‑overlapping [start, end) tuples covering the index span."""
    step = pd.Timedelta(minutes=minutes)
    start = idx.min()
    windows = []
    while start < idx.max():
        end = start + step
        windows.append((start, end))
        start = end
    return windows


# ── MAIN PIPELINE ───────────────────────────────────────────────────────────
def main() -> None:
    # 1. Feature matrix X (tickers × OFI) & target y (self returns)
    X_full = load_cross_ofi(DATA_DIR)
    y_full = load_self_returns(PRICE_CSV)

    # Align; keep timestamps present in both
    df_all = X_full.join(y_full, how="inner")
    y = df_all.pop("y")
    X = df_all

    # 2. Rolling LASSO selection
    results = []
    windows = rolling_windows(X.index, WINDOW_MIN)

    for win_start, win_end in windows:
        mask = (X.index >= win_start) & (X.index < win_end)
        if mask.sum() < 10:               # need enough samples
            continue

        X_win = X.loc[mask]
        y_win = y.loc[mask]

        model = LassoCV(
            cv=5,
            random_state=LASSO_STATE,
            max_iter=10000,
        ).fit(X_win, y_win)

        nz_idx = np.flatnonzero(model.coef_)
        sel_tickers = list(X.columns[nz_idx])
        sel_coefs   = model.coef_[nz_idx].round(6).tolist()

        results.append({
            "window_start": win_start,
            "window_end":   win_end,
            "self_ticker":  SELF_TICKER,
            "selected_tickers": sel_tickers,
            "coef_values":      sel_coefs,
        })

    # 3. Save selection log
    pd.DataFrame(results).to_csv(OUT_CSV, index=False)
    print(f"✅ Cross‑asset selections written to {OUT_CSV}")


if __name__ == "__main__":
    main()
